<a href="https://colab.research.google.com/github/bagchibijoyini-a11y/cerebro_2026/blob/main/prob2_hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# STUDENT EMOTIONAL STATE DETECTION FROM RAW AUDIO
# Hackathon Prototype
#
# Features:
#   1. Pitch
#   2. Energy
#   3. Speech rate
#   4. Pauses
#   5. MFCC
#   6. Zero Crossing Rate
#   7. Spectral Centroid
#   8. Spectral Bandwidth
#
# Machine Learning:
#   Random Forest Classifier
#
# Input:
#   Student audio (.wav / .mp3 / .flac)
#
# Output:
#   Predicted emotional state + confidence
# ================================================================


# ================================================================
# STEP 1: INSTALL LIBRARIES
# ================================================================

!pip -q install librosa soundfile numpy pandas matplotlib scikit-learn joblib


# ================================================================
# STEP 2: IMPORT LIBRARIES
# ================================================================

import os
import warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import soundfile as sf
import joblib

from google.colab import files
from IPython.display import Audio, display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score
)

warnings.filterwarnings("ignore")

print("=" * 70)
print("STUDENT EMOTIONAL STATE DETECTION FROM RAW AUDIO")
print("=" * 70)


# ================================================================
# STEP 3: FUNCTION TO EXTRACT AUDIO FEATURES
# ================================================================

def extract_audio_features(file_path):

    # ------------------------------------------------------------
    # Load audio
    # ------------------------------------------------------------

    y, sr = librosa.load(
        file_path,
        sr=16000,
        mono=True
    )

    if len(y) == 0:
        raise ValueError("Audio file is empty.")

    duration = len(y) / sr

    # ------------------------------------------------------------
    # 1. PITCH
    # ------------------------------------------------------------

    try:

        f0, voiced_flag, voiced_prob = librosa.pyin(
            y,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr
        )

        pitch_values = f0[~np.isnan(f0)]

        if len(pitch_values) > 0:

            mean_pitch = np.mean(pitch_values)
            pitch_std = np.std(pitch_values)
            min_pitch = np.min(pitch_values)
            max_pitch = np.max(pitch_values)

        else:

            mean_pitch = 0
            pitch_std = 0
            min_pitch = 0
            max_pitch = 0

    except:

        mean_pitch = 0
        pitch_std = 0
        min_pitch = 0
        max_pitch = 0


    # ------------------------------------------------------------
    # 2. ENERGY
    # ------------------------------------------------------------

    rms = librosa.feature.rms(y=y)[0]

    mean_energy = np.mean(rms)
    energy_std = np.std(rms)

    max_energy = np.max(rms)


    # ------------------------------------------------------------
    # 3. ZERO CROSSING RATE
    # ------------------------------------------------------------

    zcr = librosa.feature.zero_crossing_rate(y)[0]

    mean_zcr = np.mean(zcr)
    zcr_std = np.std(zcr)


    # ------------------------------------------------------------
    # 4. SPECTRAL CENTROID
    # ------------------------------------------------------------

    spectral_centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=sr
    )[0]

    mean_centroid = np.mean(spectral_centroid)
    centroid_std = np.std(spectral_centroid)


    # ------------------------------------------------------------
    # 5. SPECTRAL BANDWIDTH
    # ------------------------------------------------------------

    spectral_bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=sr
    )[0]

    mean_bandwidth = np.mean(spectral_bandwidth)
    bandwidth_std = np.std(spectral_bandwidth)


    # ------------------------------------------------------------
    # 6. PAUSE DETECTION
    # ------------------------------------------------------------

    intervals = librosa.effects.split(
        y,
        top_db=30
    )

    speech_duration = 0

    for start, end in intervals:

        speech_duration += (
            end - start
        ) / sr

    pause_duration = max(
        0,
        duration - speech_duration
    )

    number_of_pauses = max(
        0,
        len(intervals) - 1
    )

    pause_ratio = (
        pause_duration / duration
        if duration > 0
        else 0
    )


    # ------------------------------------------------------------
    # 7. SPEECH RATE
    # ------------------------------------------------------------

    onset_env = librosa.onset.onset_strength(
        y=y,
        sr=sr
    )

    onset_frames = librosa.onset.onset_detect(
        onset_envelope=onset_env,
        sr=sr
    )

    onset_count = len(onset_frames)

    speech_rate = (
        onset_count / duration
        if duration > 0
        else 0
    )


    # ------------------------------------------------------------
    # 8. MFCC
    # ------------------------------------------------------------

    mfcc = librosa.feature.mfcc(
        y=y,
        sr=sr,
        n_mfcc=13
    )

    mfcc_mean = np.mean(
        mfcc,
        axis=1
    )

    mfcc_std = np.std(
        mfcc,
        axis=1
    )


    # ------------------------------------------------------------
    # 9. CHROMA
    # ------------------------------------------------------------

    chroma = librosa.feature.chroma_stft(
        y=y,
        sr=sr
    )

    chroma_mean = np.mean(
        chroma,
        axis=1
    )


    # ------------------------------------------------------------
    # 10. COMBINE ALL FEATURES
    # ------------------------------------------------------------

    basic_features = np.array([

        mean_pitch,
        pitch_std,
        min_pitch,
        max_pitch,

        mean_energy,
        energy_std,
        max_energy,

        mean_zcr,
        zcr_std,

        mean_centroid,
        centroid_std,

        mean_bandwidth,
        bandwidth_std,

        duration,

        speech_duration,
        pause_duration,

        number_of_pauses,
        pause_ratio,

        speech_rate

    ])

    feature_vector = np.concatenate([

        basic_features,

        mfcc_mean,
        mfcc_std,

        chroma_mean

    ])

    return feature_vector


# ================================================================
# STEP 4: FEATURE NAMES
# ================================================================

feature_names = [

    "Mean_Pitch",
    "Pitch_Std",
    "Min_Pitch",
    "Max_Pitch",

    "Mean_Energy",
    "Energy_Std",
    "Max_Energy",

    "Mean_ZCR",
    "ZCR_Std",

    "Mean_Spectral_Centroid",
    "Spectral_Centroid_Std",

    "Mean_Spectral_Bandwidth",
    "Spectral_Bandwidth_Std",

    "Duration",

    "Speech_Duration",
    "Pause_Duration",

    "Number_of_Pauses",
    "Pause_Ratio",

    "Speech_Rate"
]


feature_names += [
    f"MFCC_{i}_Mean"
    for i in range(1, 14)
]

feature_names += [
    f"MFCC_{i}_Std"
    for i in range(1, 14)
]

feature_names += [
    f"Chroma_{i}"
    for i in range(1, 13)
]


print("\nNumber of extracted features:",
      len(feature_names))


# ================================================================
# STEP 5: UPLOAD TRAINING DATASET
# ================================================================

print("\n" + "=" * 70)
print("TRAINING DATASET")
print("=" * 70)

print("""
Your training dataset should preferably be organized like this:

audio_dataset/
│
├── calm/
│     ├── audio1.wav
│     ├── audio2.wav
│
├── stressed/
│     ├── audio3.wav
│     ├── audio4.wav
│
├── frustrated/
│     ├── audio5.wav
│     ├── audio6.wav
│
├── sad/
│
├── happy/
│
└── neutral/

You can upload a ZIP file containing this folder structure.
""")


print("Upload your ZIP dataset now.")
print("If you already have the dataset in Colab, you can skip this upload.")

uploaded_dataset = files.upload()


# ================================================================
# STEP 6: EXTRACT ZIP DATASET
# ================================================================

import zipfile

dataset_path = "/content/audio_dataset"

zip_files = [
    f for f in uploaded_dataset.keys()
    if f.lower().endswith(".zip")
]

if len(zip_files) > 0:

    zip_file = zip_files[0]

    print("\nExtracting:", zip_file)

    with zipfile.ZipFile(
        zip_file,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            "/content/"
        )

    # Find emotion folders automatically

    possible_folders = [
        "/content/audio_dataset",
        "/content/dataset",
        "/content/data"
    ]

    found = False

    for folder in possible_folders:

        if os.path.exists(folder):

            dataset_path = folder
            found = True
            break

    if not found:

        # Search directories

        for root, dirs, files_list in os.walk(
            "/content"
        ):

            audio_dirs = []

            for d in dirs:

                dir_path = os.path.join(
                    root,
                    d
                )

                audio_files = [
                    f for f in os.listdir(
                        dir_path
                    )
                    if f.lower().endswith(
                        (".wav", ".mp3", ".flac")
                    )
                ]

                if len(audio_files) > 0:

                    audio_dirs.append(d)

            if len(audio_dirs) >= 2:

                dataset_path = root
                found = True
                break

    print(
        "Dataset location:",
        dataset_path
    )

else:

    print(
        "\nNo ZIP dataset detected."
    )


# ================================================================
# STEP 7: BUILD TRAINING DATASET
# ================================================================

X = []
y_labels = []

if os.path.exists(dataset_path):

    print("\nScanning dataset...")

    for emotion in os.listdir(
        dataset_path
    ):

        emotion_path = os.path.join(
            dataset_path,
            emotion
        )

        if not os.path.isdir(
            emotion_path
        ):
            continue

        audio_files = [
            f for f in os.listdir(
                emotion_path
            )
            if f.lower().endswith(
                (".wav", ".mp3", ".flac")
            )
        ]

        if len(audio_files) == 0:
            continue

        print(
            f"\nEmotion: {emotion}"
        )

        print(
            "Audio files:",
            len(audio_files)
        )

        for file in audio_files:

            file_path = os.path.join(
                emotion_path,
                file
            )

            try:

                features = extract_audio_features(
                    file_path
                )

                X.append(features)
                y_labels.append(emotion)

            except Exception as e:

                print(
                    "Error:",
                    file,
                    e
                )


# ================================================================
# STEP 8: CONVERT DATA TO NUMPY
# ================================================================

if len(X) > 0:

    X = np.array(X)
    y_labels = np.array(y_labels)

    print("\n" + "=" * 70)
    print("DATASET INFORMATION")
    print("=" * 70)

    print(
        "Number of audio samples:",
        X.shape[0]
    )

    print(
        "Number of features:",
        X.shape[1]
    )

    print(
        "Emotion classes:",
        np.unique(y_labels)
    )

    print("\nClass distribution:")

    print(
        pd.Series(y_labels)
        .value_counts()
    )


# ================================================================
# STEP 9: TRAIN MACHINE LEARNING MODEL
# ================================================================

if len(X) >= 10:

    print("\n" + "=" * 70)
    print("TRAINING RANDOM FOREST MODEL")
    print("=" * 70)

    X_train, X_test, y_train, y_test = train_test_split(

        X,
        y_labels,

        test_size=0.20,

        random_state=42,

        stratify=y_labels
    )


    # ------------------------------------------------------------
    # Feature scaling
    # ------------------------------------------------------------

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )


    # ------------------------------------------------------------
    # Random Forest
    # ------------------------------------------------------------

    model = RandomForestClassifier(

        n_estimators=200,

        random_state=42,

        class_weight="balanced",

        n_jobs=-1

    )


    model.fit(
        X_train_scaled,
        y_train
    )


    # ============================================================
    # STEP 10: TEST MODEL
    # ============================================================

    y_pred = model.predict(
        X_test_scaled
    )


    accuracy = accuracy_score(
        y_test,
        y_pred
    )


    print("\nModel Accuracy:",
          round(accuracy * 100, 2),
          "%")


    # ============================================================
    # CLASSIFICATION REPORT
    # ============================================================

    print("\nClassification Report:\n")

    print(
        classification_report(
            y_test,
            y_pred
        )
    )


    # ============================================================
    # CONFUSION MATRIX
    # ============================================================

    plt.figure(
        figsize=(8, 6)
    )

    ConfusionMatrixDisplay.from_predictions(

        y_test,

        y_pred,

        xticks_rotation=45

    )

    plt.title(
        "Student Emotion Classification"
    )

    plt.tight_layout()

    plt.show()


    # ============================================================
    # SAVE MODEL
    # ============================================================

    joblib.dump(
        model,
        "/content/emotion_model.pkl"
    )

    joblib.dump(
        scaler,
        "/content/audio_scaler.pkl"
    )

    print(
        "\nModel saved successfully."
    )


else:

    print("\n" + "=" * 70)
    print("TRAINING MODEL SKIPPED")
    print("=" * 70)

    print("""
A labelled training dataset was not found.

You need at least a collection of labelled audio recordings
organized into emotion folders.

Example:

audio_dataset/
    calm/
    stressed/
    frustrated/
    sad/
    happy/
    neutral/

The system will then train the Random Forest classifier.
""")


# ================================================================
# STEP 11: UPLOAD NEW STUDENT AUDIO
# ================================================================

print("\n" + "=" * 70)
print("UPLOAD STUDENT AUDIO FOR PREDICTION")
print("=" * 70)

print("""
Upload one audio recording of a student.

Supported formats:
.wav
.mp3
.flac
""")

uploaded_audio = files.upload()


audio_files = [
    f for f in uploaded_audio.keys()
    if f.lower().endswith(
        (".wav", ".mp3", ".flac")
    )
]


if len(audio_files) == 0:

    print(
        "No supported audio file uploaded."
    )

else:

    test_audio = audio_files[0]

    print(
        "\nSelected audio:",
        test_audio
    )


    # ============================================================
    # STEP 12: PLAY AUDIO
    # ============================================================

    print("\nPlaying audio:")

    display(
        Audio(test_audio)
    )


    # ============================================================
    # STEP 13: LOAD AUDIO FOR ANALYSIS
    # ============================================================

    y_audio, sr_audio = librosa.load(

        test_audio,

        sr=16000,

        mono=True
    )


    duration_audio = (
        len(y_audio) /
        sr_audio
    )


    # ============================================================
    # STEP 14: DISPLAY WAVEFORM
    # ============================================================

    plt.figure(
        figsize=(14, 4)
    )

    librosa.display.waveshow(
        y_audio,
        sr=sr_audio
    )

    plt.title(
        "Student Speech Waveform"
    )

    plt.xlabel(
        "Time (seconds)"
    )

    plt.ylabel(
        "Amplitude"
    )

    plt.show()


    # ============================================================
    # STEP 15: EXTRACT FEATURES
    # ============================================================

    test_features = extract_audio_features(
        test_audio
    )


    print("\n" + "=" * 70)
    print("EXTRACTED AUDIO FEATURES")
    print("=" * 70)


    feature_table = pd.DataFrame(

        test_features.reshape(1, -1),

        columns=feature_names

    )


    display(
        feature_table.T
    )


    # ============================================================
    # STEP 16: SHOW IMPORTANT ACOUSTIC FEATURES
    # ============================================================

    print("\n" + "=" * 70)
    print("IMPORTANT ACOUSTIC CHARACTERISTICS")
    print("=" * 70)


    mean_pitch = test_features[0]

    pitch_variation = test_features[1]

    mean_energy = test_features[4]

    pause_duration = test_features[15]

    number_pauses = test_features[16]

    speech_rate = test_features[18]


    print(
        "Mean Pitch:",
        round(mean_pitch, 2),
        "Hz"
    )

    print(
        "Pitch Variation:",
        round(pitch_variation, 2)
    )

    print(
        "Mean Energy:",
        round(mean_energy, 4)
    )

    print(
        "Pause Duration:",
        round(pause_duration, 2),
        "seconds"
    )

    print(
        "Number of Pauses:",
        int(number_pauses)
    )

    print(
        "Estimated Speech Rate:",
        round(speech_rate, 2),
        "events/sec"
    )


    # ============================================================
    # STEP 17: MACHINE LEARNING PREDICTION
    # ============================================================

    if "model" in globals() and "scaler" in globals():

        test_features_scaled = scaler.transform(

            test_features.reshape(
                1,
                -1
            )
        )


        prediction = model.predict(

            test_features_scaled
        )[0]


        probabilities = model.predict_proba(

            test_features_scaled
        )[0]


        classes = model.classes_


        probability_df = pd.DataFrame({

            "Emotion": classes,

            "Probability (%)":
                probabilities * 100

        })


        probability_df = (
            probability_df
            .sort_values(
                "Probability (%)",
                ascending=False
            )
            .reset_index(
                drop=True
            )
        )


        probability_df[
            "Probability (%)"
        ] = probability_df[
            "Probability (%)"
        ].round(2)


        # ========================================================
        # STEP 18: FINAL RESULT
        # ========================================================

        confidence = (
            np.max(probabilities)
            * 100
        )


        print("\n")

        print("=" * 70)

        print(
            "              FINAL AI RESULT"
        )

        print("=" * 70)


        print(
            "\nPredicted Emotional State:"
        )

        print(
            ">>>",
            prediction.upper()
        )


        print(
            "\nModel Confidence:"
        )

        print(
            ">>>",
            round(confidence, 2),
            "%"
        )


        print(
            "\nEmotion Probability:"
        )

        display(
            probability_df
        )


        # ========================================================
        # STEP 19: INTERPRETATION
        # ========================================================

        interpretations = {

            "stressed":
            "The acoustic pattern is associated with elevated stress indicators.",

            "frustrated":
            "The acoustic pattern is associated with frustration or dissatisfaction indicators.",

            "sad":
            "The acoustic pattern is associated with a subdued emotional state.",

            "happy":
            "The acoustic pattern is associated with a positive emotional state.",

            "calm":
            "The acoustic pattern is associated with a relatively calm emotional state.",

            "neutral":
            "The recording does not show strong evidence for the other modeled emotional categories.",

            "anxious":
            "The acoustic pattern is associated with anxiety-related emotional indicators."

        }


        print(
            "\nInterpretation:"
        )

        print(
            interpretations.get(
                prediction,
                "An emotional state was detected."
            )
        )


        # ========================================================
        # STEP 20: HACKATHON SAFETY NOTE
        # ========================================================

        print("\n" + "-" * 70)

        print(
            "IMPORTANT:"
        )

        print(
            "This system detects acoustic emotional indicators."
        )

        print(
            "It is NOT a clinical or medical diagnosis."
        )

        print(
            "Predictions depend on the quality and diversity "
            "of the training dataset."
        )

        print("-" * 70)


    else:

        # ========================================================
        # NO TRAINED MODEL
        # ========================================================

        print("\n" + "=" * 70)

        print(
            "FEATURE EXTRACTION COMPLETED"
        )

        print("=" * 70)

        print("""
The audio features were successfully extracted.

However, no trained emotion-classification model is available.

The next step is to provide a labelled emotion dataset and
train the Random Forest model.

The extracted features include:

• Pitch
• Pitch variation
• Energy
• Speech rate
• Pause duration
• Number of pauses
• MFCC
• Zero Crossing Rate
• Spectral Centroid
• Spectral Bandwidth
• Chroma

These features can then be used to classify emotional states.
""")


# ================================================================
# END
# ================================================================

print("\n" + "=" * 70)
print("PROGRAM COMPLETED")
print("=" * 70)

STUDENT EMOTIONAL STATE DETECTION FROM RAW AUDIO

Number of extracted features: 57

TRAINING DATASET

Your training dataset should preferably be organized like this:

audio_dataset/
│
├── calm/
│     ├── audio1.wav
│     ├── audio2.wav
│
├── stressed/
│     ├── audio3.wav
│     ├── audio4.wav
│
├── frustrated/
│     ├── audio5.wav
│     ├── audio6.wav
│
├── sad/
│
├── happy/
│
└── neutral/

You can upload a ZIP file containing this folder structure.

Upload your ZIP dataset now.
If you already have the dataset in Colab, you can skip this upload.
